# Dataset Information

## Glass Identification Dataset

Conjunto de dados de classificação multiclasse com 214 amostras de vidro, cada uma descrita por seu índice de refração e pela composição química. O objetivo é prever o tipo de vidro a partir dessas medidas.

O problema vem da ciência forense: vidro deixado em uma cena de crime pode servir de evidência, desde que seja identificado corretamente. Saber se um fragmento veio de uma janela de prédio, de um farol de carro ou de um recipiente ajuda a ligar suspeitos ao local.

Características gerais
Item	Valor
Amostras	214
Atributos preditivos	9 (todos numéricos contínuos)
Atributo-alvo	Tipo de vidro (6 classes presentes)
Valores ausentes	Nenhum
Tarefa	Classificação
Atributos
##	Atributo	Descrição
1.	    Id	Identificador da amostra (1 a 214).
2.	    RI	Índice de refração
3.	    Na	Sódio
4.	    Mg	Magnésio
5.	    Al	Alumínio
6.	    Si	Silício
7.	    K	Potássio
8.	    Ca	Cálcio
9.	    Ba	Bário
10.	    Fe	Ferro
11.	    Type	Tipo de vidro (classe-alvo)

Os atributos de 3 a 10 são dados em percentual em peso do óxido correspondente (por exemplo, o Na representa Na₂O).

## Classe Alvo
Esta pode ser de 6 tipos:

Código  |   Tipo de vidro   |   Amostras
1.	    Janela de construção, float	
2.	    Janela de construção, não float	
3.	    Janela de veículo, float	
5.	    Recipientes	
6.	    Louça de mesa	
7. 	    Faróis	

Fonte: https://archive.ics.uci.edu/dataset/42/glass+identification


## importanto o dataset

In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Carregamento do dataset
glass_identification = fetch_ucirepo(id=42)

# Separação das features (X) e da variável alvo (y)
X = glass_identification.data.features
y = glass_identification.data.targets.squeeze()

# Visualização dos dados
display(X)
display(y)

# Verificação de valores ausentes
print("Total de valores ausentes nas features:", X.isnull().sum().sum())
print("Total de valores ausentes na variável alvo:", y.isnull().sum())

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0
...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0


0      1
1      1
2      1
3      1
4      1
      ..
209    7
210    7
211    7
212    7
213    7
Name: Type_of_glass, Length: 214, dtype: int64

Total de valores ausentes nas features: 0
Total de valores ausentes na variável alvo: 0


## Pré-processamento dos dados

Antes do treinamento dos modelos, foi realizada a preparação do conjunto de dados. A variável `Type`, correspondente ao tipo de vidro, foi definida como variável alvo (`y`), enquanto as demais colunas foram utilizadas como características de entrada (`X`).

Foi verificada a presença de valores ausentes no conjunto de dados e não foram encontrados valores nulos, portanto não foi necessário realizar procedimentos de preenchimento ou remoção de dados.

Também não foi necessária a aplicação de técnicas de codificação, como One-Hot Encoding ou Label Encoding, nas características de entrada, pois todas as features utilizadas pelos modelos são numéricas.

O conjunto de dados foi dividido em **80% para treinamento e 20% para teste**. Foi utilizado o parâmetro `stratify=y` para preservar, tanto quanto possível, a proporção das diferentes classes nos conjuntos de treinamento e teste. Também foi definido `random_state=42` para permitir a reprodução dos experimentos.

## Partição dos dados em treino e teste

### Holdout 80/20

In [2]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Amostras de treinamento:", len(X_tr))
print("Amostras de teste:", len(X_te))

Amostras de treinamento: 171
Amostras de teste: 43


## Árvore de Decisão

In [5]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import graphviz

# Parâmetros que serão testados pelo GridSearch
grade_arvore = {
    "max_depth": [3, 4, 5, None],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": [None, "balanced"]
}

# Busca pelos melhores hiperparâmetros
busca_arvore = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    grade_arvore,
    cv=5,
    scoring="f1_macro"
)

busca_arvore.fit(X_tr, y_tr)

# Melhor modelo encontrado
modelo_arvore = busca_arvore.best_estimator_

print("Melhores parâmetros:")
print(busca_arvore.best_params_)

# Previsões no conjunto de teste
pred_arvore = modelo_arvore.predict(X_te)

print("\nRelatório de classificação:")
print(
    classification_report(
        y_te,
        pred_arvore,
        zero_division=0
    )
)

# Gerar visualização da árvore
dot = export_graphviz(
    modelo_arvore,
    feature_names=X.columns,
    class_names=[str(c) for c in modelo_arvore.classes_],
    filled=True,
    rounded=True,
    impurity=False,
    label="none",
    special_characters=True
)

g = graphviz.Source(
    dot.replace(
        "digraph Tree {",
        "digraph Tree {\nrankdir=LR;",
        1
    )
)

# Salvar árvore em SVG
g.format = "svg"
g.render("arvore", cleanup=True)

# Exibir árvore no notebook
g

Melhores parâmetros:
{'class_weight': 'balanced', 'max_depth': 4, 'min_samples_leaf': 1}

Relatório de classificação:
              precision    recall  f1-score   support

           1       0.75      0.43      0.55        14
           2       0.62      0.33      0.43        15
           3       0.25      1.00      0.40         3
           5       0.50      0.67      0.57         3
           6       0.67      1.00      0.80         2
           7       0.50      0.67      0.57         6

    accuracy                           0.51        43
   macro avg       0.55      0.68      0.55        43
weighted avg       0.62      0.51      0.51        43



ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

## Naive Bayes

In [ ]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

glass = fetch_ucirepo(id=42)

X = glass.data.features
y = glass.data.targets.squeeze()

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

modelo_nb = GaussianNB().fit(X_tr, y_tr)

print(classification_report(
    y_te,
    modelo_nb.predict(X_te),
    zero_division=0
))

In [ ]:


print("Probabilidades das classes:")
display(pd.DataFrame({
    "classe": modelo_nb.classes_,
    "probabilidade": modelo_nb.class_prior_
}))

print("Médias das features por classe:")
display(pd.DataFrame(
    modelo_nb.theta_,
    index=modelo_nb.classes_,
    columns=X.columns
).round(3))

In [ ]:
amostra = X_te.iloc[[0]]

print("Classe real:", y_te.iloc[0])
print("Classe prevista:", modelo_nb.predict(amostra)[0])

display(pd.DataFrame({
    "classe": modelo_nb.classes_,
    "probabilidade": modelo_nb.predict_proba(amostra)[0]
}).sort_values("probabilidade", ascending=False))

## KNN

In [ ]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

glass = fetch_ucirepo(id=42)
X = glass.data.features
y = glass.data.targets.squeeze()

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Normalização dos dados para o KNN

Como o KNN realiza suas classificações com base na distância entre as amostras, diferenças de escala entre as features podem fazer com que determinadas variáveis exerçam uma influência desproporcional no cálculo das distâncias.

Por esse motivo, foi utilizado o `StandardScaler` para padronizar as features antes da aplicação do KNN. A padronização foi incorporada a um `Pipeline`, garantindo que o ajuste do `StandardScaler` seja realizado utilizando os dados de treinamento e aplicado posteriormente aos dados de teste, evitando vazamento de informações do conjunto de teste durante o treinamento.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

pipe = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())])
grade = {"knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15], "knn__weights": ["uniform", "distance"], "knn__p": [1, 2]}
busca = GridSearchCV(pipe, grade, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1_macro")
modelo = busca.fit(X_tr, y_tr)

print(modelo.best_params_)
print(classification_report(y_te, modelo.predict(X_te), zero_division=0))

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
r = cross_validate(busca, X, y, cv=cv, scoring=["accuracy", "f1_macro"], n_jobs=-1)
print(*[f"{m}: {r['test_'+m].mean():.3f} ± {r['test_'+m].std():.3f}" for m in ["accuracy", "f1_macro"]])

pred = cross_val_predict(busca, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=-1)
print(classification_report(y, pred, zero_division=0))
print(confusion_matrix(y, pred))

In [ ]:
# Interpretabilidade do KNN

melhor_knn = modelo.best_estimator_
amostra = X_te.iloc[[0]]

k = melhor_knn.named_steps["knn"].n_neighbors
peso = melhor_knn.named_steps["knn"].weights

p = melhor_knn.named_steps["knn"].p

amostra_escalada = melhor_knn.named_steps["sc"].transform(amostra)

distancias, indices = melhor_knn.named_steps["knn"].kneighbors(
    amostra_escalada,
    n_neighbors=k
)

classe_real = y_te.iloc[0]
classe_prevista = modelo.predict(amostra)[0]

print("Classe real:", classe_real)
print("Classe prevista:", classe_prevista)
print("Número de vizinhos (k):", k)
print("Tipo de peso:", peso)
print("Métrica p:", p)

print("\nAmostra analisada:")
print(amostra)

print("\nVizinhos mais próximos:")

for posicao, (indice, distancia) in enumerate(
    zip(indices[0], distancias[0]),
    start=1
):
    print(f"\nVizinho {posicao}")
    print("Índice no treino:", indice)
    print("Classe:", y_tr.iloc[indice])
    print("Distância:", distancia)
    print("Características:")
    print(X_tr.iloc[indice])

# Classes dos vizinhos
classes_vizinhos = y_tr.iloc[indices[0]]

print("\nClasses dos vizinhos:")
print(classes_vizinhos.values)

print("\nContagem das classes:")
print(classes_vizinhos.value_counts())

# Se estiver usando peso por distância, mostra também o peso de cada vizinho
if peso == "distance":
    pesos_vizinhos = 1 / (distancias[0] + 1e-10)

    print("\nPesos dos vizinhos:")
    for i, peso_vizinho in enumerate(pesos_vizinhos, start=1):
        print(f"Vizinho {i}: {peso_vizinho}")

# Resultado final
if classe_real == classe_prevista:
    print("\nResultado: previsão correta.")
else:
    print("\nResultado: previsão incorreta.")